# **Problem Statement**

  NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

  To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

# **Objective**

* **Provide real-time, personalized news retrieval** to help users discover relevant content effortlessly.
* **Ensure accuracy and credibility** by sourcing news from trusted platforms and minimizing misinformation.
* **Improve user engagement** through seamless content discovery, reducing information overload.
* **Streamline the news consumption process** by eliminating outdated and irrelevant content, providing a refined reading experience.

--------------------------------

# **Installing necessary packages**

In [ ]:
%pip install -qU langchain-groq
%pip install -qU \
    langchain==0.3.27 \
    langchainhub==0.1.21 \
    langchain-experimental==0.3.4 \
    langchain-groq==0.3.8 \
    ddgs==9.6.1 \
    numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.3

In [ ]:
import json
import os
import pandas as pd
import sqlite3
import joblib

from langchain.agents import create_sql_agent, initialize_agent # Added initialize_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool

from ddgs import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

Groq API Key

In [ ]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GroqCloud_APIkey')

In [ ]:
"""
from google.colab import drive
drive.mount('/content/drive')
"""

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n"

In [ ]:
from langchain_groq import ChatGroq  # Import Groq LLM

# Get the API key from Colab secrets
groq_api_key = userdata.get('GroqCloud_APIkey')

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    groq_api_key=groq_api_key
)

In [ ]:
#from google.colab import userdata
from langchain_groq import ChatGroq  # Import Groq LLM

# Get the API key from Colab secrets
groq_api_key = userdata.get('GroqCloud_APIkey') #Complete the code by calling the API key

# High creativity LLM
llm_high = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Call the Groq model
    temperature=1.5, #Complete the code by providing high temperature
    groq_api_key=groq_api_key)

# Low creativity (deterministic) LLM
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Call the Groq model
    temperature=0, #Complete the code by providing low temperature
    groq_api_key=groq_api_key
)

# **Criteria 01**

    Setting Up the LLM
      - Install the required libraries
      - Load an LLM using Groq
      - Check the LLM response on a simple query

        Points
        4


In [ ]:
for i in range(2):
  response = llm.predict("How can I use a 'Deterministic' (Temperature 0) LLM to verify the facts generated by a 'Creative' (Temperature 1.5) LLM?")
  print(response)
  print("-"*50)

To verify the facts generated by a 'Creative' (Temperature 1.5) LLM using a 'Deterministic' (Temperature 0) LLM, you can follow these steps:

**Understanding the role of Temperature in LLMs:**
- **Temperature 0 (Deterministic):** At this setting, the model will produce outputs that are essentially the most likely or highest probability next piece of text. This setting tends to generate more factual, repetitive, and sometimes conservative responses. It's less likely to produce novel or creative outputs but is good for tasks requiring accuracy and determinism.
- **Temperature 1.5 (Creative):** A higher temperature increases the randomness of the output, allowing the model to explore less probable but potentially more creative or novel responses. This setting is useful for generating diverse ideas, creative writing, or when you want the model to take more risks in its responses.

**Verification Process:**
1. **Generate Content with Creative LLM (Temperature 1.5):** Use the creative model 

In [ ]:
for i in range(2):
  response = llm_high.predict("How can I use a 'Deterministic' (Temperature 0) LLM to verify the facts generated by a 'Creative' (Temperature 1.5) LLM?")
  print(response)
  print("-"*50)

Using a deterministic (Temperature 0) LLM to verify the facts generated by a 'creative' (Temperature 1.5) LLM is an interesting approach to ensuring the accuracy and reliability of the generated content. Here's a step-by-step guide on how to implement this:

**Understanding LLM Temperature**

Before diving into the verification process, it's essential to understand the role of temperature in LLMs. Temperature is a hyperparameter that controls the randomness of the model's output. A temperature of 0 makes the model deterministic, meaning it will always produce the same output given the same input. A higher temperature (e.g., 1.5) makes the model more creative and diverse in its outputs, but also more prone to errors.

**Verification Process**

1. **Generate creative content**: Use the 'creative' LLM (Temperature 1.5) to generate content, such as text, answers, or summaries. This will produce diverse and potentially creative outputs.
2. **Extract claims and facts**: From the generated co

# **Criteria 02**

    SQL Agent for Data Retrieval
     - Initialize the SQL agent to fetch the data from the database
     - Initialize the system message
     - Verify the customer's email and retrieve their details

       Points
       14

In [ ]:
db = SQLDatabase.from_uri("sqlite:////content/customer.db")

#Initialize the system message
system_message = "You are an SQL assistant. Your role is to efficiently fetch and retrieve user data from the database."

#Initialize the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

#Create the SQL agent with the system message
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)

In [ ]:
query = f"Fetch all the unique email_id"


# Execute the query using the SQL agent and store the output
output = sql_agent.invoke(query)

# Display the retrieved records
output

{'input': 'Fetch all the unique email_id',
 'output': 'Here are the unique email_ids:\n\n1. alice.6eb33c45-5@gmail.com\n2. alice.d56e53b6-1@gmail.com\n3. emma.a88fec03-c@gmail.com\n4. george.5483cb53-8@gmail.com\n5. hannah.4770b814-2@gmail.com\n6. ian.203631a0-b@gmail.com\n7. ian.aee37571-7@gmail.com\n8. julia.b5a14512-c@gmail.com\n9. julia.cf4d1edb-9@gmail.com\n10. julia.d77d96f3-3@gmail.com\n11. kevin.f8641860-7@gmail.com\n12. lily.45e01919-0@gmail.com\n13. lily.fa8fe05b-7@gmail.com\n14. nora.c63fa237-e@gmail.com\n15. oscar.edd38e10-6@gmail.com \n\nHowever, as per the instructions, I should limit the results to at most 10. Here is the revised answer:\n\n1. alice.6eb33c45-5@gmail.com\n2. alice.d56e53b6-1@gmail.com\n3. emma.a88fec03-c@gmail.com\n4. george.5483cb53-8@gmail.com\n5. hannah.4770b814-2@gmail.com\n6. ian.203631a0-b@gmail.com\n7. ian.aee37571-7@gmail.com\n8. julia.b5a14512-c@gmail.com\n9. julia.cf4d1edb-9@gmail.com\n10. julia.d77d96f3-3@gmail.com'}

In [ ]:
query = f"Fetch all the unique customer details"


# Execute the query using the SQL agent and store the output
output = sql_agent.invoke(query)

# Display the retrieved records
output

{'input': 'Fetch all the unique customer details',
 'output': 'Here are the unique customer details:\n\n- Customer ID: F8641860-7, Name: Kevin, Email: kevin.f8641860-7@gmail.com\n- Customer ID: 203631A0-B, Name: Ian, Email: ian.203631a0-b@gmail.com\n- Customer ID: D77D96F3-3, Name: Julia, Email: julia.d77d96f3-3@gmail.com\n- Customer ID: 6EB33C45-5, Name: Alice, Email: alice.6eb33c45-5@gmail.com\n- Customer ID: EDD38E10-6, Name: Oscar, Email: oscar.edd38e10-6@gmail.com\n- Customer ID: 4770B814-2, Name: Hannah, Email: hannah.4770b814-2@gmail.com\n- Customer ID: 5483CB53-8, Name: George, Email: george.5483cb53-8@gmail.com\n- Customer ID: FA8FE05B-7, Name: Lily, Email: lily.fa8fe05b-7@gmail.com\n- Customer ID: C63FA237-E, Name: Nora, Email: nora.c63fa237-e@gmail.com\n- Customer ID: 45E01919-0, Name: Lily, Email: lily.45e01919-0@gmail.com'}

providing the email id which is not present in the database to check the response of LLm

In [ ]:
email="rifakath341@gmail.com"
query= f"Fetch all records with email_id {email}"

output=sql_agent.invoke(query)

output

{'input': 'Fetch all records with email_id rifakath341@gmail.com',
 'output': 'No records found with email_id "rifakath341@gmail.com".'}

providing the email id which is present in the database to check the response of LLm


In [ ]:
email="alice.6eb33c45-5@gmail.com"

query= f"Fetch all the Interests with email_id {email} in a list"

output=sql_agent.invoke(query)

output

{'input': 'Fetch all the Interests with email_id alice.6eb33c45-5@gmail.com in a list',
 'output': '["Politics", "Technology", "Business"]'}

In [ ]:
email="alice.d56e53b6-1@gmail.com"

query= f"Fetch all the Interests with email_id {email} in a list"

output=sql_agent.invoke(query)

output

{'input': 'Fetch all the Interests with email_id alice.d56e53b6-1@gmail.com in a list',
 'output': '["International", "Technology"]'}

In [ ]:
"""
db = SQLDatabase.from_uri("sqlite:////content/customer.db")

#Initialize the system message
system_message = "You are an SQL assistant. Your role is to efficiently fetch and retrieve user data from the database."

#Initialize the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

#Create the SQL agent with the system message
sql_agent1 = create_sql_agent(
    llm=llm_high,
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)
"""

In [ ]:
"""
query = f"Fetch the customer's email and retrieve their details"


# Execute the query using the SQL agent and store the output
output = sql_agent1.invoke(query)

# Display the retrieved records
output
"""

{'input': "Fetch the customer's email and retrieve their details",
 'output': 'Here are the customer details including their email:\n\n1. Customer ID: 203631A0-B, Name: Ian, Email: ian.203631a0-b@gmail.com, Interests: ["Startups", "Travel"]\n2. Customer ID: 45E01919-0, Name: Lily, Email: lily.45e01919-0@gmail.com, Interests: ["India", "India", "Politics"]\n3. Customer ID: 4770B814-2, Name: Hannah, Email: hannah.4770b814-2@gmail.com, Interests: ["Entertainment", "Business", "Technology"]\n4. Customer ID: 5483CB53-8, Name: George, Email: george.5483cb53-8@gmail.com, Interests: ["India", "Politics", "Science"]\n5. Customer ID: 6EB33C45-5, Name: Alice, Email: alice.6eb33c45-5@gmail.com, Interests: ["Politics", "Technology", "Business"]\n6. Customer ID: A88FEC03-C, Name: Emma, Email: emma.a88fec03-c@gmail.com, Interests: ["Technology", "Automobile"]\n7. Customer ID: AEE37571-7, Name: Ian, Email: ian.aee37571-7@gmail.com, Interests: ["Entertainment", "Automobile", "Startups"]\n8. Customer ID

# **Criteria 03**

    Interface between SQL and Search Agents
      - Create an expanded search query to search for precise news and the latest information
      - Fetch the news results using DuckDuckGo
      - Filter the relevant and trustworthy news URLs based on the users' interests

        Points
        14

Generating Expanded Search Queries for Current News

In [ ]:
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    # Handle dict input
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query","")
    else:
        # Handle string input fallback
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""

    system_prompt = """
You are a News Discovery Expert for NewsFindr. Your goal is to convert broad user interests and specific queries into precise, time-sensitive search terms for a news search engine.

Rules for Query Generation:
1. Prioritize Recency: Include keywords like 'latest', 'today', 'breaking news', or '2026' to ensure results are current.
2. Maximize Precision: Combine the 'interest' and 'user_query' into a single, focused search string.
3. Search Engine Optimization: Use keywords that news crawlers favor (e.g., 'developments', 'press release', 'report').
4. Conciseness: Keep the query under 10 words.
5. Output Format: Provide ONLY the search query text. No explanations, no quotes, and no preambles.
"""                                                                       #Complete the code by providing a clear instructions in the system prompt

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries

# Define the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)

Fetch News Results Using DuckDuckGo

In [ ]:
def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            title = r.get("title", "")
            url = r.get("href", "")
            body = r.get("body", "")
            res={"title":title,"url":url,"body":body}
            results.append(res)
    return json.dumps(results)
ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)

Filter Relevant and Trustworthy URLs Based on User Interests

In [ ]:
def filter_with_llm(search_results: list) -> list:

    search_results= json.dumps(search_results, indent=2)
    system_prompt = """
You are the Lead Credibility Analyst for NewsFindr. Your mission is to protect users from misinformation and information overload.

Your task is to analyze search results and filter them based on:
1. Source Reliability: Favor established news agencies (e.g., AP, Reuters, BBC), academic journals, and official reports.
2. Neutrality: Avoid sensationalist clickbait, extreme bias, or unverified blog posts.
3. Relevance: Ensure the content is strictly news-related and matches the context of the user's request.
4. Recency: Prioritize current events from 2026.

Your output must be a clean, high-quality selection of news that builds user trust.
      """

    prompt = f"""
Below is a list of raw search results in JSON format. Please audit these results and select the top 3 most trustworthy and relevant articles.

For each selected article, return:
- The Title
- The URL
- A 1-sentence 'Credibility Verdict' explaining why this source was chosen.

If a result appears to be a 'fake news' site or low-quality content, exclude it entirely.

Search Results to Audit:
{search_results}
"""

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content


credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)

Generate summary for the URLs

In [ ]:
def summarize_news(url_list) -> dict:
    """
    Summarizes key news points from a list of URLs and returns both the summary and the URLs.
    Since only URLs are available, the LLM will generate a high-level summary
    based on the website context or domain.
    """

    system_prompt = """
You are a Senior News Editor for NewsFindr. Your goal is to synthesize multiple news sources into a single, cohesive, and easy-to-digest summary.

Your responsibilities:
1. Conciseness: Distill long articles into their most critical facts.
2. Structure: Use clear headings or bullet points for readability.
3. Tone: Maintain a professional, objective, and engaging tone.
4. Relevance: Focus on 'Breaking News' elements and how they impact the user.
5. Trust: Only summarize the information likely to be found at these reputable sources; do not hallucinate details.
"""

    # Pass the URLs to the LLM for summarization
    prompt = f"""
Please provide a high-level summary and key takeaways from the following list of news URLs.
Synthesize the information so that a user can understand the current landscape of these topics in under 30 seconds.

Sources to summarize:
"""+ "\n".join(url_list)

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )

    summary_text = response.content

    # Return a dictionary containing the summary and the original list of URLs
    return {"summary": summary_text, "urls": url_list}


summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)

Creating an Agent

In [ ]:
#Final code
tools = [expand_tool, ddg_search_tool, credibility_tool, summarize_tool]                                                    #Complete the code by calling the already defined functions

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

Display Final Verified URLs and Summary based on the User interest and User query

In [ ]:
#Final code
email = "hannah.4770b814-2@gmail.com"
interest_result = sql_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

# Normalize to list
if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
    interests = interest_result["output"]["items"]
elif isinstance(interest_result["output"], str):
    interests = [i.strip() for i in interest_result["output"].split(",")]
else:
    interests = interest_result["output"]

# Define the user query
user_query = "latest business trends in the entertainment industry driven by AI technology"

# Agent prompt stays the same, just note we’re passing the dict
agent_prompt = f"""
I have a user with the following interests: {interests}.
The user has a specific request: '{user_query}'.

Please perform the following steps:
1. Use the ExpandSearchQueries tool to create 2026-specific search queries based on these interests and the user request.
2. Use DuckDuckGoSearch to find the latest news articles.
3. Use the CredibilityFilter to ensure the sources are trustworthy and relevant.
4. Finally, use the SummarizeNews tool to provide a clean, bulleted report.
"""
response = agent.run(agent_prompt)
print("\n======= FINAL RESPONSE =======")
print(response)




> Entering new AgentExecutor chain...
To address the user's request, I need to follow the steps outlined, starting with expanding the search queries based on the user's interests and specific request.

Action: ExpandSearchQueries
Action Input: ['["Entertainment"', '"Business"', '"Technology"]', 'latest business trends in the entertainment industry driven by AI technology']
Observation: ['latest entertainment news today breaking developments', 'latest business news today global market developments report', 'latest tech developments and innovations today', 'latest AI technology trends in entertainment industry business']
Thought:

I now have a list of expanded search queries that are specific to the user's interests and request. The next step is to use DuckDuckGoSearch to find the latest news articles based on these queries.

Action: DuckDuckGoSearch
Action Input: 'latest AI technology trends in entertainment industry business'
Observation: [{"title": "Fox News - Breaking News Updates | Latest News Headlines | Photos ...", "url": "https://www.foxnews.com/?msockid=2353780401ff63c719636f4a0042626d", "body": "Breaking News, Latest News and Current News from FOXNews.com. Breaking news and video. Latest Current News: U.S., World, Entertainment, Health, Business, Technology ..."}, {"title": "Breaking News, Latest News and Videos | CNN", "url": "https://www.cnn.com/", "body": "View the latest news and breaking news today for U.S., world, weather, entertainment, politics and health at CNN.com."}, {"title": "Associated Press News : Breaking News , Latest Headlines and Videos | AP News", "url": "https://apnews.com/", "bod

# **Criteria 04**

    Output from LLMs

    - Use the LLM to Retrieve the final URL(s) for the latest news based on the customer’s interest
    - Create a summary of the retrieved links

      Points
      10

Retrieve URLs and Summaries for Three Areas of Interest

In [ ]:
def query_response(email: str, user_query: str) -> str:
    # Fetch interests based on the provided email
    interest_result = sql_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

    # Normalize to list
    if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
      interests = interest_result["output"]["items"]
    elif isinstance(interest_result["output"], str):
        interests = [i.strip() for i in interest_result["output"].split(",")]
    else:
       interests = interest_result["output"]


    # Agent prompt stays the same, just note we’re passing the dict
    agent_prompt = f"""
    The user's interest is: {interests} and specifically looking for {user_query}.
    Here is the process to follow:
    1. Expand the user's interest into news search queries using the 'ExpandSearchQueries' tool.
       The input to this tool should be a dictionary like: {{'interests': {interests}, 'user_query': '{user_query}'}}.
    2. Use the query generated in step 1 to search for recent news using the 'DuckDuckGoSearch' tool for all the queries.
    3. Pass the combined results of all queries into 'CredibilityFilter' for filtering.
    4. From the filtered results, select the top 3 latest news articles for summarization.
    5. Summarize all credible results into a detailed summary using the 'SummarizeNews'.
    6. Provide the final summary to the user.
    """
    response = agent.run(agent_prompt)
    print("\n======= FINAL RESPONSE =======")
    print(response)

    return response

In [ ]:
#Check for the user ID
email = "emma.a88fec03-c@gmail.com"
user_query = "Find the latest news on self-driving car technology advancements and how AI is being integrated into the next generation of electric vehicles."
response = query_response(email, user_query)



> Entering new AgentExecutor chain...
Thought: The first step is to expand the user's interest into precise, time-sensitive news search queries using the 'ExpandSearchQueries' tool. The input should be a dictionary containing the user's interests and the specific query.

Action: ExpandSearchQueries
Action Input: {'interests': ['["Technology"', '"Automobile"]'], 'user_query': 'Find the latest news on self-driving car technology advancements and how AI is being integrated into the next generation of electric vehicles.'}
Observation: ['latest technology news and developments today', 'latest automobile industry developments today', 'latest self-driving car technology advancements AI electric vehicles']
Thought:Thought: Now that we have the expanded search queries, the next step is to use the 'DuckDuckGoSearch' tool to search for recent news for each of these queries. This will help us gather relevant news articles.

Action: DuckDuckGoSearch
Action Input: 'latest self-driving car technolo

In [ ]:
#Check for the user ID
email = "george.5483cb53-8@gmail.com"
user_query = "Search for the latest scientific breakthroughs in India's research sectors and provide an update on any recent political policies or government funding changes impacting the science community."
response = query_response(email, user_query)



> Entering new AgentExecutor chain...
Thought: The first step is to expand the user's interest into precise, time-sensitive news search queries using the 'ExpandSearchQueries' tool. The input to this tool should be a dictionary containing the user's interests and the specific query.

Action: ExpandSearchQueries
Action Input: {'interests': ['["India"', '"Politics"', '"Science"]'], 'user_query': 'Search for the latest scientific breakthroughs in India\'s research sectors and provide an update on any recent political policies or government funding changes impacting the science community.'}
Observation: ['latest news in India today', 'latest political developments today in US government', 'latest science breakthroughs and discoveries today', 'latest scientific breakthroughs India research sectors government funding policies']
Thought:Thought: I have obtained a list of expanded search queries. The next step is to use the 'DuckDuckGoSearch' tool to search for recent news based on these que

In [ ]:
#Check for the user ID
email = "hannah.4770b814-2@gmail.com"
user_query = "Search for credible reports on how major entertainment studios are restructuring their business models in 2026 to leverage generative AI for film production."
response = query_response(email, user_query)



> Entering new AgentExecutor chain...
Thought: The first step is to expand the user's interest into precise, time-sensitive news search queries using the 'ExpandSearchQueries' tool. The input should be a dictionary containing the user's interests and the specific query.

Action: ExpandSearchQueries
Action Input: {'interests': ['["Entertainment"', '"Business"', '"Technology"]'], 'user_query': 'Search for credible reports on how major entertainment studios are restructuring their business models in 2026 to leverage generative AI for film production.'}
Observation: ['latest entertainment news today breaking developments', 'latest business news today global market developments report', 'latest tech developments and innovations today', 'latest entertainment industry AI film production business models 2026 developments report']
Thought:Thought: Now that we have expanded search queries, the next step is to use these queries to search for recent news using the 'DuckDuckGoSearch' tool. We wil

# **Conclusion**

 **Personalized Retrieval**:
    
* The sql_agent queries the customer.db to identify the specific interests of a user (e.g., Emma’s interest in Technology and Automobile).  

**Query Expansion**:
*  The expand_tool takes these broad interests and the user’s specific query to generate precise, time-sensitive search terms for the current year, 2026.

**Real-Time Search**:
* The ddg_search_tool executes these queries to fetch the top 5 most relevant live links from the web.

**Credibility Filtering**:
* The credibility_tool acts as a fact-checker, auditing the URLs to prioritize authoritative sources like Reuters or the BBC while removing clickbait or misinformation.

**Summarization**:
*  Finally, the summarize_tool synthesizes the information from the trusted sources into a concise, bulleted report.